In [1]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from IPython.display import Image, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from src.config import *
RES = RES_DIR
def show(name, width=900):
    display(Image(filename=str(FIG_DIR / f"{name}.png"), width=width))


# 01 · Data understanding

**Question.** Can we load the three Elliptic files, join them reliably and describe what we have?

Elliptic (Weber et al., 2019) is a public Bitcoin graph released by the blockchain-analytics firm Elliptic:
each **node** is a transaction, each **edge** is a flow of coins from one transaction into another, and each node has
166 anonymised features (the first is the time step, leaving 165 model inputs: 93 *local* + 72 *aggregated-neighbour*).

Files: `elliptic_txs_features.csv` (no header), `elliptic_txs_classes.csv` (`1` illicit, `2` licit, `unknown`), `elliptic_txs_edgelist.csv`.

In [2]:
from src.data_loader import load_master, load_raw, build_master
feats, classes, edges = load_raw()
print("features:", feats.shape, "| classes:", classes.shape, "| edges:", edges.shape)
master, edges = load_master()           # joins + verifies ids (assertions inside build_master)
print("master:", master.shape)
master.iloc[:3, :6]

features: (203769, 167) | classes: (203769, 2) | edges: (234355, 2)


master: (203769, 168)


,txId,time_step,local_1,local_2,local_3,local_4
0,230425980,1,-0.171469,-0.184668,-1.201369,-0.12197
1,5530458,1,-0.171484,-0.184668,-1.201369,-0.12197
2,232022460,1,-0.172107,-0.184668,-1.201369,-0.12197


**Join verification (all done by assertions in `build_master`).** Transaction ids are unique in both the feature and class files,
the two id sets are identical, the join is one-to-one (203,769 rows in, 203,769 rows out), and every edge endpoint exists in the
feature table. Raw label codes were recoded to `1 = illicit, 0 = licit, -1 = unknown` so that "2" is no longer mistaken for the positive class.

In [3]:
from src.preprocessing import quality_report
q = quality_report(master, edges)
pd.Series({k: v for k, v in q.items() if k != "outlier_share_per_feature"})

n_transactions                                                            203769
n_edges                                                                   234355
n_features                                                                   165
n_time_steps                                                                  49
missing_values_total                                                           0
duplicate_txids                                                                0
duplicate_feature_rows                                                        34
duplicate_edges                                                                0
self_loop_edges                                                                0
edges_within_same_time_step                                                  1.0
constant_features                                                             []
dtypes                         {'float32': 165, 'int64': 1, 'int16': 1, 'int8...
inf_values                  

## Data-quality decisions

| # | Problem found | Why it matters | Action taken | Why that is appropriate |
|---|---|---|---|---|
| 1 | **157,205 transactions (77.1%) have no label** | A model cannot learn from, or be scored on, unknown ground truth. Treating unknown as licit would inject label noise and flatter the metrics. | Kept in the master table, but **excluded from every train/validation/test set and every supervised metric** (`labelled()` helper). Unknown rows are used only for label-free work: graph degrees and input-drift (KS) measurement. | Keeps evaluation honest; uses the extra data only where no label is required. |
| 2 | **Missing values: 0** (and 0 infinite values) | Would break most models if present. | No imputation was needed on this data; a median imputer fitted on training rows is still part of every pipeline. | Makes the prediction pipeline robust to uploaded CSVs with gaps, without leaking test statistics. |
| 3 | **Duplicate ids: 0, duplicate edges: 0, self-loops: 0. 34 duplicate *feature rows*** (49 rows in 15 groups) | Duplicates could leak between train and test. | Checked: all 15 groups sit **inside the same time step** and **none has conflicting labels**. Rows kept. | They are distinct transactions with identical features (0.017% of rows); they cannot cross the chronological split, so there is no leakage and deleting them would discard real events. |
| 4 | **Heavy tails / outliers**: median feature has 13.4% of values beyond 3×IQR (max 38.3%) | Linear models are distorted by extreme values; but extremes may be genuine behaviour. | **Not removed.** Tree models are scale-invariant. For Logistic Regression: standardise (train statistics) then clip at ±5 SD. | In the boxplot analysis (notebook 02) extremes are overwhelmingly *licit* transactions, so deleting outliers would delete signal and bias the data. |
| 5 | **Data types**: features arrive as float64/int64 | 690 MB CSV; memory. | Features cast to `float32`, `time_step` to `int16`, `label` to `int8`; the headerless feature file is given explicit names (`local_1…93`, `agg_1…72`). | 4× smaller without losing precision relevant to trees/LR. Column names make downstream code and explanations readable. |
| 6 | **Time step is column 2 of the feature file** | If used as an input the model could memorise "when" instead of "what", and could not generalise to unseen future steps. | Used **only** to split the data and to evaluate per step; never a model feature. | Prevents a form of leakage and matches real deployment. |
| 7 | **Edges never cross time steps** (share within the same step = 1.0) | If edges linked to the future, graph features would leak later information. | Verified; so degree and neighbour features are computed per step without using future data. | The graph is effectively 49 separate snapshots. |
| 8 | **Class imbalance**: 9.8% of labelled transactions are illicit | Accuracy is misleading. | Class-weighting / `scale_pos_weight`, illicit-class precision/recall/F1/PR-AUC, majority baseline reported. | See notebook 04. |

In [4]:
lab = master["label"].map({1: "illicit", 0: "licit", -1: "unknown"}).value_counts()
summary = pd.DataFrame({"count": lab, "share": (lab / lab.sum()).round(4)})
summary.loc["labelled illicit share"] = [np.nan, round(4545 / (4545 + 42019), 4)]
summary

,count,share
label,,
unknown,157205.0,0.7715
licit,42019.0,0.2062
illicit,4545.0,0.0223
labelled illicit share,NaN,0.0976


**Summary.** 203,769 transactions, 234,355 edges, 49 time steps. 4,545 illicit (2.2%), 42,019 licit (20.6%), 157,205 unknown (77.1%).
Only the 46,564 labelled transactions (9.8% illicit) are used for supervised learning. The data is clean in the sense of no missing values or
duplicate ids; the real difficulties are imbalance, scarcity of labels, heavy tails and time (next notebooks).